In [1]:
import pandas as pd
import numpy as np
import re

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("../data/raw/dataset.csv")

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

In [3]:
def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

ce_cols = sorted(
    ce_cols,
    key=get_strike
)

pe_cols = sorted(
    pe_cols,
    key=get_strike
)

In [4]:
records = []

for idx,row in df.iterrows():

    spot = row["underlying_price"]

    for cols in [ce_cols, pe_cols]:

        values = row[cols].values

        for j,col in enumerate(cols):

            if pd.isna(row[col]):
                continue

            left_iv = np.nan
            right_iv = np.nan

            if j > 0:
                left_iv = row[cols[j-1]]

            if j < len(cols)-1:
                right_iv = row[cols[j+1]]

            records.append({

                "target": row[col],

                "strike": get_strike(col),

                "option_type":
                    0 if col.endswith("CE")
                    else 1,

                "spot": spot,

                "moneyness":
                    get_strike(col)/spot,

                "left_iv": left_iv,

                "right_iv": right_iv,

                "row_mean":
                    np.nanmean(values),

                "row_median":
                    np.nanmedian(values)

            })

train_df = pd.DataFrame(records)

print(train_df.shape)
train_df.head()

(21840, 9)


,target,strike,option_type,spot,moneyness,left_iv,right_iv,row_mean,row_median
0,0.12662,25200,0,26111.65,0.965086,NaN,0.12330,0.102447,0.09647
1,0.12330,25300,0,26111.65,0.968916,0.12662,0.11741,0.102447,0.09647
2,0.11741,25400,0,26111.65,0.972746,0.12330,NaN,0.102447,0.09647
3,0.11005,25600,0,26111.65,0.980405,NaN,0.10576,0.102447,0.09647
4,0.10576,25700,0,26111.65,0.984235,0.11005,NaN,0.102447,0.09647


In [5]:
train_df["neighbor_mean"] = (
    train_df["left_iv"] +
    train_df["right_iv"]
) / 2

train_df["neighbor_diff"] = (
    train_df["right_iv"] -
    train_df["left_iv"]
)

print(train_df.head())

    target  strike  option_type      spot  moneyness  left_iv  right_iv  \
0  0.12662   25200            0  26111.65   0.965086      NaN   0.12330   
1  0.12330   25300            0  26111.65   0.968916  0.12662   0.11741   
2  0.11741   25400            0  26111.65   0.972746  0.12330       NaN   
3  0.11005   25600            0  26111.65   0.980405      NaN   0.10576   
4  0.10576   25700            0  26111.65   0.984235  0.11005       NaN   

   row_mean  row_median  neighbor_mean  neighbor_diff  
0  0.102447     0.09647            NaN            NaN  
1  0.102447     0.09647       0.122015       -0.00921  
2  0.102447     0.09647            NaN            NaN  
3  0.102447     0.09647            NaN            NaN  
4  0.102447     0.09647            NaN            NaN  


In [6]:
features = [

    "strike",
    "option_type",
    "spot",
    "moneyness",

    "left_iv",
    "right_iv",

    "neighbor_mean",
    "neighbor_diff",

    "row_mean",
    "row_median"
]

model_df = train_df.copy()

model_df = model_df.dropna(
    subset=[
        "left_iv",
        "right_iv"
    ]
)

X = model_df[features]

y = model_df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11986, 10)
y shape: (11986,)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000743 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2067
[LightGBM] [Info] Number of data points in the train set: 9588, number of used features: 10
[LightGBM] [Info] Start training from score 0.183771
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [9]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_val)

mse = mean_squared_error(
    y_val,
    preds
)

print("Neighbor LightGBM MSE:", mse)

Neighbor LightGBM MSE: 0.0005547295950519483


In [10]:
option_cols = ce_cols + pe_cols

missing_counts_per_row = df[option_cols].isna().sum(axis=1)

print(missing_counts_per_row.describe())

print("\nValue counts:")
print(
    missing_counts_per_row
    .value_counts()
    .sort_index()
)

count    975.000000
mean       5.600000
std        2.069134
min        0.000000
25%        4.000000
50%        5.000000
75%        7.000000
max       14.000000
dtype: float64

Value counts:
0       3
1      10
2      31
3     100
4     162
5     186
6     187
7     127
8      84
9      49
10     19
11     10
12      6
14      1
Name: count, dtype: int64


In [11]:
missing_matrix = df[option_cols].isna()

missing_by_col = missing_matrix.sum()

print(
    missing_by_col.sort_values()
)

NIFTY27JAN2624700PE    175
NIFTY27JAN2624100PE    176
NIFTY27JAN2625600CE    180
NIFTY27JAN2625500CE    181
NIFTY27JAN2626000CE    181
NIFTY27JAN2625900CE    182
NIFTY27JAN2626100CE    183
NIFTY27JAN2626500CE    184
NIFTY27JAN2625300CE    184
NIFTY27JAN2626300CE    187
NIFTY27JAN2624300PE    189
NIFTY27JAN2625200CE    194
NIFTY27JAN2624500PE    195
NIFTY27JAN2625000PE    195
NIFTY27JAN2624900PE    195
NIFTY27JAN2624600PE    198
NIFTY27JAN2623900PE    201
NIFTY27JAN2625700CE    201
NIFTY27JAN2625400CE    202
NIFTY27JAN2624400PE    202
NIFTY27JAN2623800PE    203
NIFTY27JAN2625100PE    205
NIFTY27JAN2626200CE    209
NIFTY27JAN2624000PE    209
NIFTY27JAN2624800PE    209
NIFTY27JAN2625800CE    210
NIFTY27JAN2626400CE    213
NIFTY27JAN2624200PE    217
dtype: int64


In [12]:
for col in ce_cols[:5]:
    
    miss_idx = df.index[
        df[col].isna()
    ]

    print(
        col,
        miss_idx[:20].tolist()
    )

NIFTY27JAN2625200CE [6, 7, 10, 17, 22, 23, 29, 32, 35, 39, 44, 46, 52, 54, 60, 80, 86, 88, 89, 93]
NIFTY27JAN2625300CE [1, 2, 18, 25, 26, 34, 36, 38, 40, 44, 45, 51, 52, 56, 63, 75, 76, 77, 79, 83]
NIFTY27JAN2625400CE [1, 16, 17, 24, 32, 40, 46, 47, 48, 54, 57, 61, 62, 65, 70, 79, 81, 95, 97, 101]
NIFTY27JAN2625500CE [0, 7, 8, 10, 11, 13, 15, 16, 20, 23, 30, 33, 36, 48, 57, 74, 82, 85, 89, 92]
NIFTY27JAN2625600CE [5, 7, 11, 14, 15, 17, 19, 30, 33, 35, 36, 60, 67, 72, 82, 86, 89, 93, 95, 98]


In [13]:
for col in pe_cols[:5]:
    
    miss_idx = df.index[
        df[col].isna()
    ]

    print(
        col,
        miss_idx[:20].tolist()
    )

NIFTY27JAN2623800PE [7, 11, 16, 25, 26, 39, 42, 44, 55, 57, 59, 61, 65, 66, 71, 76, 80, 90, 96, 97]
NIFTY27JAN2623900PE [3, 4, 11, 12, 17, 19, 32, 44, 46, 52, 54, 55, 57, 62, 67, 69, 77, 79, 89, 93]
NIFTY27JAN2624000PE [1, 13, 14, 18, 24, 29, 30, 31, 35, 37, 39, 42, 45, 49, 51, 58, 59, 64, 66, 71]
NIFTY27JAN2624100PE [0, 14, 39, 55, 64, 65, 68, 73, 75, 76, 77, 78, 79, 86, 99, 102, 103, 104, 111, 113]
NIFTY27JAN2624200PE [1, 11, 13, 15, 16, 19, 21, 25, 34, 50, 52, 54, 58, 66, 70, 72, 77, 81, 83, 91]


In [14]:
option_cols = ce_cols + pe_cols

corrs = []

for i in range(len(option_cols)-1):

    c1 = option_cols[i]
    c2 = option_cols[i+1]

    m1 = df[c1].isna().astype(int)
    m2 = df[c2].isna().astype(int)

    corr = m1.corr(m2)

    corrs.append(corr)

print("Mean adjacent missing correlation:",
      np.mean(corrs))

print("Max adjacent missing correlation:",
      np.max(corrs))

print("Min adjacent missing correlation:",
      np.min(corrs))

Mean adjacent missing correlation: 0.001885758243548173
Max adjacent missing correlation: 0.06853109065229301
Min adjacent missing correlation: -0.05128205128205132


In [15]:
missing_matrix = df[option_cols].isna().astype(int)

row_corr = missing_matrix.T.corr()

print(
    row_corr.values.mean()
)

nan


In [16]:
option_cols = ce_cols + pe_cols

missing_positions = []

for _, row in df[option_cols].isna().iterrows():

    miss = np.where(row.values)[0]

    if len(miss) > 0:

        missing_positions.extend(
            miss.tolist()
        )

print(pd.Series(missing_positions).describe())

print()
print(
    pd.Series(missing_positions)
    .value_counts()
    .sort_index()
)

count    5460.000000
mean       13.604762
std         8.061418
min         0.000000
25%         7.000000
50%        14.000000
75%        21.000000
max        27.000000
dtype: float64

0     194
1     184
2     202
3     181
4     180
5     201
6     210
7     182
8     181
9     183
10    209
11    187
12    213
13    184
14    203
15    201
16    209
17    176
18    217
19    189
20    202
21    195
22    198
23    175
24    209
25    195
26    195
27    205
Name: count, dtype: int64


In [19]:
from scipy.interpolate import interp1d
import numpy as np

def linear_fill(values, strikes):

    values = values.copy()

    mask = ~np.isnan(values)

    if mask.sum() < 2:
        return values

    f = interp1d(
        strikes[mask],
        values[mask],
        kind="linear",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(
        strikes[~mask]
    )

    return values

In [22]:
import re
import numpy as np

def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

ce_cols = sorted(
    ce_cols,
    key=get_strike
)

pe_cols = sorted(
    pe_cols,
    key=get_strike
)

ce_strikes = np.array(
    [get_strike(c) for c in ce_cols]
)

pe_strikes = np.array(
    [get_strike(c) for c in pe_cols]
)

print("CE strikes:", ce_strikes)
print("PE strikes:", pe_strikes)

CE strikes: [25200 25300 25400 25500 25600 25700 25800 25900 26000 26100 26200 26300
 26400 26500]
PE strikes: [23800 23900 24000 24100 24200 24300 24400 24500 24600 24700 24800 24900
 25000 25100]


In [23]:
def evaluate_linear_ce_pe(seed=42):

    np.random.seed(seed)

    temp = df.copy()

    ce_hidden = []
    pe_hidden = []

    # Hide 20% CE values
    for cols, hidden_store in [
        (ce_cols, ce_hidden),
        (pe_cols, pe_hidden)
    ]:

        known = []

        for col in cols:

            valid_rows = temp.index[
                temp[col].notna()
            ]

            for r in valid_rows:
                known.append((r,col))

        known = np.array(
            known,
            dtype=object
        )

        n_hide = int(
            len(known) * 0.20
        )

        selected = np.random.choice(
            len(known),
            n_hide,
            replace=False
        )

        for idx in selected:

            r,col = known[idx]

            hidden_store.append(
                (
                    r,
                    col,
                    temp.loc[r,col]
                )
            )

            temp.loc[r,col] = np.nan

    # Linear fill CE
    for idx in temp.index:

        temp.loc[idx,ce_cols] = linear_fill(
            temp.loc[idx,ce_cols]
            .astype(float)
            .values,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = linear_fill(
            temp.loc[idx,pe_cols]
            .astype(float)
            .values,
            pe_strikes
        )

    ce_true = []
    ce_pred = []

    for r,col,val in ce_hidden:

        ce_true.append(val)
        ce_pred.append(
            temp.loc[r,col]
        )

    pe_true = []
    pe_pred = []

    for r,col,val in pe_hidden:

        pe_true.append(val)
        pe_pred.append(
            temp.loc[r,col]
        )

    ce_mse = mean_squared_error(
        ce_true,
        ce_pred
    )

    pe_mse = mean_squared_error(
        pe_true,
        pe_pred
    )

    return ce_mse, pe_mse

In [24]:
results = []

for seed in [1,2,3,4,5]:

    ce_mse, pe_mse = evaluate_linear_ce_pe(seed)

    results.append({
        "seed": seed,
        "ce_mse": ce_mse,
        "pe_mse": pe_mse
    })

results_df = pd.DataFrame(results)

print(results_df)
print()
print(results_df.mean(numeric_only=True))

   seed    ce_mse    pe_mse
0     1  0.000106  0.000169
1     2  0.000101  0.000044
2     3  0.000079  0.000117
3     4  0.000145  0.000175
4     5  0.000123  0.000037

seed      3.000000
ce_mse    0.000111
pe_mse    0.000108
dtype: float64
